In [ ]:
#@title Prevent disconnections
%%html
<audio src="https://oobabooga.github.io/silence.m4a" controls>

In [ ]:
#@title Setup SwarmUI
import os
SWARMPATH = '/content/'
os.environ['SWARMPATH'] = SWARMPATH
os.environ['SWARM_NO_VENV'] = 'true'

# Keep dotnet stable on Colab (JIT/AVX-512 VM bugs + low memory)
os.environ['DOTNET_TieredPGO'] = '0'
os.environ['DOTNET_TieredCompilation'] = '0'
os.environ['DOTNET_EnableWriteXorExecute'] = '0'
os.environ['DOTNET_EnableAVX512F'] = '0'
os.environ['DOTNET_gcServer'] = '0'
os.environ['MSBUILDDISABLENODEREUSE'] = '1'

!apt install -y aria2

# Colab has no swap; add 8G so the build doesn't crash
!if ! swapon --show | grep -q swapfile; then fallocate -l 8G /content/swapfile && chmod 600 /content/swapfile && mkswap /content/swapfile && swapon /content/swapfile; fi

# dotnet 10 SDK (required by current SwarmUI)
!wget -q https://dot.net/v1/dotnet-install.sh -O dotnet-install.sh
!chmod +x dotnet-install.sh
!./dotnet-install.sh --channel 10.0

# cloudflared for a public share URL
!wget -q https://github.com/cloudflare/cloudflared/releases/download/2024.8.2/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

%cd $SWARMPATH

# Clone SwarmUI (skip if already cloned)
!if [ ! -d /content/SwarmUI/.git ]; then git clone https://github.com/mcmonkeyprojects/SwarmUI.git; fi

# Create model directory
!mkdir -p /content/SwarmUI/Models/checkpoints

In [ ]:
#@title Select & Download Model
KEY = ""  # @param {type:"string"}
KEY = "token=" + KEY.strip()

MODELS = {
    "deepDarkHentaiMixNSFW_v61Hybrid": {
        "file": "deepDarkHentaiMixNSFW_v61Hybrid.safetensors",
        "url": f"https://civitai.com/api/download/models/634653?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
    "cyberrealisticPony_semiRealV40": {
        "file": "cyberrealisticPony_semiRealV40.safetensors",
        "url": f"https://civitai.com/api/download/models/2268768?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
    "novaCartoon_v10": {
        "file": "novaCartoon_v10.safetensors",
        "url": f"https://civitai.com/api/download/models/821389?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
}

CHOICE = "cyberrealisticPony_semiRealV40"  # @param ["cyberrealisticPony_semiRealV40","deepDarkHentaiMixNSFW_v61Hybrid","novaCartoon_v10"]

MODEL = MODELS[CHOICE]
MODEL_PATH = "/content/SwarmUI/Models/checkpoints"
USER_AGENT = '"User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64)"'

!aria2c --enable-http-keep-alive=false --header=$USER_AGENT --console-log-level=error -c -x 16 -s 16 -k 1M --summary-interval=5 -d $MODEL_PATH -o {MODEL['file']} "{MODEL['url']}"

In [ ]:
#@title Launch SwarmUI
import os
# Keep the dotnet stability flags in case the kernel restarted since setup
os.environ['DOTNET_TieredPGO'] = '0'
os.environ['DOTNET_TieredCompilation'] = '0'
os.environ['DOTNET_EnableWriteXorExecute'] = '0'
os.environ['DOTNET_EnableAVX512F'] = '0'
os.environ['DOTNET_gcServer'] = '0'
os.environ['MSBUILDDISABLENODEREUSE'] = '1'

%cd /content/SwarmUI

# Patch launchtools/linux-build-logic.sh to make its dotnet build Colab-safe
logic = 'launchtools/linux-build-logic.sh'
old = 'dotnet build src/SwarmUI.csproj --configuration Release -o ./src/bin/live_release'
new = old + ' -m:1 --disable-build-servers'
with open(logic) as f:
    content = f.read()
if old in content:
    with open(logic, 'w') as f:
        f.write(content.replace(old, new))
    print('Patched build command (added -m:1 --disable-build-servers)')
else:
    print('Patch already applied or pattern not found')

!rm -rf ./src/bin/live_release

!bash ./launch-linux.sh --launch_mode none --cloudflared-path cloudflared